In [ ]:
# Used Claude Code to help implement the experiment to see how paramater recovert changes with noise

In [1]:
# One thread per process, set before torch is imported.
#
# The sweep runs several inversions at once, one process each. A multi-threaded
# BLAS inside each of them would oversubscribe the machine — every process takes
# the ten cores it can see, so the threads multiply by the workers — and
# contention costs more than the threads buy on a problem this size, where the
# arrays are small enough that the solve is memory-bound rather than
# compute-bound. torch reads these when its OpenMP runtime loads, which happens
# at import, so they have to be set before the import and not in the sweep cell.
#
# It also makes the arithmetic deterministic. A threaded reduction sums in
# whatever order the threads finish in, so the same inversion run at two thread
# counts can differ in the last bits and, through a line search, in more than the
# last bits. Pinned to one thread the sweep reproduces exactly, and a run made
# here is bit-for-bit the run made serially.
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import scipy
import torch
from joblib import Parallel, delayed

import neuralphysics
from neuralphysics import (
    LIMB,
    SOURCE_SPECS,
    CoefficientHierarchy,
    ElectrodeArray,
    Observation,
    add_measurement_noise,
    configure,
    forward,
    gaussian_dipole,
    grid_at,
    invert,
    parameter_names,
    sample_initial_parameters,
    thicknesses,
    true_sigma,
    unpack,
    viz,
)

In [2]:
print(configure())
torch.set_num_threads(1)   # the parent solves the clean field; same thread count, same arithmetic

print(f"neuralphysics {neuralphysics.__version__} | torch {torch.__version__} | "
      f"numpy {np.__version__} | scipy {scipy.__version__} | python {platform.python_version()}")
print(f"default dtype {torch.get_default_dtype()} | torch threads {torch.get_num_threads()} | "
      f"{platform.platform()}")
print(f"cores {os.cpu_count()} | OMP_NUM_THREADS {os.environ['OMP_NUM_THREADS']}")

{'dtype': 'torch.float64', 'seed': 42}
neuralphysics 0.1.0 | torch 2.5.1 | numpy 1.26.4 | scipy 1.17.1 | python 3.11.15
default dtype torch.float64 | torch threads 1 | macOS-26.6.1-arm64-arm-64bit
cores 10 | OMP_NUM_THREADS 1


In [3]:
# the forward experiment
# anything not named here is at the package default
# L = 1, z_zeta = 1, coarsening floor of 9, jacobi damping factor omega = 2/3, nu = (2, 2)
# 50 base iterations, and 100% electrode coverage on the four lateral faces

N = 65 # nodes per axis; N = 2^k + 1 must hold for the v-cycle to work
NUM_CYCLES = 8 # fixed budget - not a convergence tolerance
WIDTH = 0.05 # the 1/e blob radius of a source, a physical length

In [4]:
# the sweep: a noise-free control, then three noise levels with five draws at each

# The companion notebook covers 0.25% to 1%; this one continues the same experiment
# above it, at 2%, 3% and 4%, from the same start and against the same clean field.
# Ascending, and the control first, so the tables read down from the floor towards
# the noisiest data and a trend is a direction on the page rather than something to
# be reconstructed from the labels.
NOISE_PERCENTS = (0.0, 2.0, 3.0, 4.0)

# Five draws per noisy level. These are seed *bases*: the two sources need
# independent noise, so source s of draw j is seeded at len(sources) * j + s, which
# lays the seeds out without any two of them colliding. Base 0 gives seeds (0, 1),
# which is what NOISE_SEED = 0 gives in end-to-end-experiment.ipynb - so the first
# draw at each level is that notebook's data at this notebook's noise level, and the
# two are comparable run for run rather than only in aggregate.
NOISE_SEEDS = (0, 1, 2, 3, 4)

# Zero noise has exactly one realisation: adding a zero draw to the clean field
# gives the clean field, whatever the seed. Running five of them would be five
# copies of one answer, and the spread reported over them would be a zero meaning
# "there is only one draw" rather than "the draws agreed".
CONTROL_SEEDS = (0,)


def draws_for(percent):
    """The noise draws to run at `percent`."""
    return CONTROL_SEEDS if percent == 0.0 else NOISE_SEEDS


N_CASES = sum(len(draws_for(percent)) for percent in NOISE_PERCENTS)
print(f"{N_CASES} runs: "
      + ", ".join(f"{len(draws_for(p))} at {p:g}%" for p in NOISE_PERCENTS))

16 runs: 1 at 0%, 5 at 2%, 5 at 3%, 5 at 4%


In [5]:
# the inverse problem

# The starting point is run 1 of the ensemble from averaged-runs.ipynb, reproduced
# from that notebook's seed rather than copied across as numbers: a transcribed
# start silently stops matching the moment the draw changes, and the point of
# starting here is that these runs and that one begin in the same place. The four
# constants below are that notebook's and have to stay its.
#
# The outermost radius is not drawn, because skin thickness is assumed known and so
# is not recovered; it is a clinically invariant quantity in our experiment.
ENSEMBLE_SEED = 12345      # seeds the draws, and nothing else
N_RUNS = 5                 # how many were drawn there. The draws are sequential, so
                           # this must match for run 1 to be the same vector
LOG_SPREAD = np.log(3.0)   # conductivities drawn within a factor of three
RADIUS_SPREAD_MM = 1.5     # radii drawn within 1.5 mm

RUN = 1                    # which of those draws every case in this sweep starts from


MAX_ITERS = 200


FTOL = 1e-16
GTOL = 1e-12

RESULTS = Path("..") / "experiment-results"   # where the report picks the outputs up

# Every output of this notebook is named through this. The companion notebook writes
# the same filenames for its own noise range, and two sweeps writing one name is a
# silent overwrite whose loser is whichever ran first - so the range is carried in
# the name rather than in a memory of which was run last.
#
# A suffix rather than a prefix, so the two versions of a table sort next to each
# other and a listing groups by what the table is rather than by which sweep made
# it. The range spelled out rather than "above 1%", because 1% is itself a level of
# the companion sweep and a boundary that sits inside one of the two ranges names
# neither of them; p is the decimal point and pc is per cent, because a literal "."
# in a stem confuses suffix handling and a literal "%" starts a comment in LaTeX,
# which would truncate any \includegraphics path that carried one.
OUTPUT_SUFFIX = "-2-4pc"


LIMB_RADIUS_MM = 24.0   # the physical limb radius, at r/R = 0.5
MM_PER_UNIT = LIMB_RADIUS_MM / 0.5   # normalised radius -> millimetres

In [6]:
# three processes. Five was tried first and exhausted this machine's memory, so the
# ceiling is measured and not chosen; two then ran without trouble, and this sweep
# steps back up one to find where the real limit sits. Three is the last count that
# still fits inside the four performance cores, so if it holds it is the fastest
# setting available and not merely a survivable one.
#
# The failure mode is losing the whole run and the machine with it, so this is worth
# watching for the first few minutes rather than starting and walking away. If it
# does fall over, two is known good - drop back rather than trying to split the
# difference.
N_WORKERS = 3

# per process, measured at N = 65 over repeated evaluations of the objective, on an
# otherwise idle machine. Reported here as a floor: the number below is what the
# solve costs and not what a worker costs, and the difference is what five workers
# ran into.
MEASURED_PEAK_GB = 3.2

print(f"{N_CASES} cases over {N_WORKERS} workers "
      f"-> {-(-N_CASES // N_WORKERS)} batches of at most {N_WORKERS}")
print(f"solve alone needs {N_WORKERS * MEASURED_PEAK_GB:.1f} GB of "
      f"{os.sysconf('SC_PHYS_PAGES') * os.sysconf('SC_PAGE_SIZE') / 1e9:.1f} GB; "
      f"the rest is headroom for everything that is not the solve")

16 cases over 3 workers -> 6 batches of at most 3
solve alone needs 9.6 GB of 25.8 GB; the rest is headroom for everything that is not the solve


In [7]:
# the tissue table - read off the model
print(f"{'layer':8s} {'r/R':>13s} {'thickness':>12s} {'sigma_T':>9s} {'sigma_L':>9s}   source")
r_lo = 0.0
for nm, r_hi in zip(LIMB.layer_names, LIMB.interfaces + (0.5,)):
    sT, sL = LIMB.sigma_layers[nm]
    print(f"{nm:8s} {f'{r_lo:.3f}-{r_hi:.3f}':>13s} "
          f"{(r_hi - r_lo) * LIMB_RADIUS_MM / 0.5:9.2f} mm {sT:9.2f} {sL:9.2f}   "
          f"{'PROVISIONAL' if nm in LIMB.provisional else 'Farina, Mesin, Martina & Merletti (2004)'}")
    r_lo = r_hi

print(f"\nanisotropic layers: {sorted(LIMB.anisotropic_layers)}")
print(f"interface width  epsilon = {LIMB.epsilon:g} of L = {LIMB.epsilon * (N - 1):.2f} h at N = {N}")
print(f"geometry derived from Maksymenko et al. (2023); a {LIMB_RADIUS_MM:.0f} mm limb")

layer              r/R    thickness   sigma_T   sigma_L   source
bone       0.000-0.146      7.00 mm      0.02      0.02   Farina, Mesin, Martina & Merletti (2004)
muscle     0.146-0.417     13.00 mm      0.10      0.50   Farina, Mesin, Martina & Merletti (2004)
fat        0.417-0.479      3.00 mm      0.05      0.05   Farina, Mesin, Martina & Merletti (2004)
skin       0.479-0.500      1.00 mm      1.00      1.00   Farina, Mesin, Martina & Merletti (2004)

anisotropic layers: ['muscle']
interface width  epsilon = 0.00390625 of L = 0.25 h at N = 65
geometry derived from Maksymenko et al. (2023); a 24 mm limb


In [8]:
# the clean data: one grid, two sources, one noiseless field per source, solved once
# at the truth. Every case takes its data from these, so the twenty-one runs differ
# in their noise realisation and in nothing else - not in the grid, not in the
# sources, and not in the field the noise is added to. The control run is this field
# untouched, which is what makes it a control and not a twenty-second experiment.

grid = grid_at(N)

sources = [gaussian_dipole(grid, LIMB, width=WIDTH, r_blob_norm=r_norm, angle_deg=angle)
           for r_norm, angle in SOURCE_SPECS]

coeff_true = CoefficientHierarchy.from_vectors(LIMB, N)
phi_clean = [forward(I, coeff_true, num_cycles=NUM_CYCLES) for I in sources]

for s, phi in enumerate(phi_clean):
    rms = float(phi.pow(2).mean().sqrt())
    print(f"source {s}: clean field RMS {rms:.3e}, "
          + ", ".join(f"{p:g}% -> std {p / 100 * rms:.3e}" for p in NOISE_PERCENTS))

source 0: clean field RMS 5.161e-04, 0% -> std 0.000e+00, 2% -> std 1.032e-05, 3% -> std 1.548e-05, 4% -> std 2.064e-05
source 1: clean field RMS 4.928e-04, 0% -> std 0.000e+00, 2% -> std 9.857e-06, 3% -> std 1.479e-05, 4% -> std 1.971e-05


In [9]:
# the starting point, drawn exactly as averaged-runs.ipynb draws it and then indexed
# rather than re-drawn: the runs come out of one generator in sequence, so drawing
# N_RUNS of them and taking the RUN-th is that notebook's run 1 and not merely
# something near it. Both halves of theta come from the draw - the five
# log-conductivities and the two logits - so the geometry and the conductivities both
# start where run 1 started.

starts = sample_initial_parameters(LIMB, N_RUNS, seed=ENSEMBLE_SEED,
                                   log_spread=LOG_SPREAD,
                                   radius_spread=RADIUS_SPREAD_MM / MM_PER_UNIT)
theta_start = starts[RUN - 1]

param_names = list(parameter_names(LIMB))
layer_names = list(LIMB.layer_names[:-1])   # skin thickness is fixed, so not recovered

sigma_start = theta_start[:len(param_names)].exp()
_, _, R_start = unpack(theta_start, LIMB)

print(f"run {RUN} of {N_RUNS}, drawn at seed {ENSEMBLE_SEED}\n")
print(f"{'':10s}" + "".join(f"{nm:>10s}" for nm in param_names))
print(f"{'start':10s}" + "".join(f"{float(v):10.4f}" for v in sigma_start) + "   S/m")
print(f"{'true':10s}" + "".join(f"{float(v):10.4f}" for v in true_sigma(LIMB)) + "   S/m")
print()
print(f"{'':10s}" + "".join(f"{nm:>10s}" for nm in layer_names))
print(f"{'start':10s}"
      + "".join(f"{float(v):10.2f}" for v in thicknesses(R_start, scale=MM_PER_UNIT)[:-1]) + "   mm")
print(f"{'true':10s}"
      + "".join(f"{float(v):10.2f}"
                for v in thicknesses(LIMB.interfaces, scale=MM_PER_UNIT)[:-1]) + "   mm")

run 1 of 5, drawn at seed 12345

                bone  muscle_T  muscle_L       fat      skin
start         0.0110    0.0669    0.9610    0.0736    0.7872   S/m
true          0.0200    0.1000    0.5000    0.0500    1.0000   S/m

                bone    muscle       fat
start           6.50     13.80      2.71   mm
true            7.00     13.00      3.00   mm


In [10]:
def invert_one(percent, seed, theta0, phi_clean):
    """One case: add noise at `percent` under `seed`, then invert from `theta0`.

    `percent = 0` is the control. Nothing branches on it: `add_measurement_noise`
    returns the field it was given, the observation is the clean field, and the
    inversion that follows is the same call the other twenty make. That is the point
    of running it through here rather than special-casing it - the control differs
    from an experiment in its data and in nothing else, including the code path.

    Runs in a worker process, so it takes everything it needs as an argument or
    rebuilds it: `grid_at` is cached per process, and the sources are analytic, so
    the first case a worker takes pays for the grid and the rest do not. What is
    *not* rebuilt is the clean field - that is passed in, because re-solving it per
    worker would make the five noise levels five solves of the same problem rather
    than one solve corrupted five ways.

    The thread pin is repeated here rather than assumed. The environment variables
    set at the top of the notebook are inherited by the workers, but the count torch
    actually uses is a runtime setting, and a run whose arithmetic depends on how the
    pool was configured is not reproducible.

    Returns a dict and not a `RunResult`: the geometry history is a few hundred
    snapshots a case, it is only ever drawn as a cross-section, and this notebook
    draws no cross-sections - so sending it back through the pool would be twenty-one
    copies of something nothing here reads.
    """
    configure()
    torch.set_num_threads(1)

    grid = grid_at(N)
    sources = [gaussian_dipole(grid, LIMB, width=WIDTH, r_blob_norm=r_norm, angle_deg=angle)
               for r_norm, angle in SOURCE_SPECS]
    mask_nodes = ElectrodeArray().mask(N)   # full coverage of the four lateral faces

    observations, snr = [], []
    for s, phi in enumerate(phi_clean):
        phi_obs, stats = add_measurement_noise(phi, percent, len(phi_clean) * seed + s)
        observations.append(Observation(phi=phi_obs, mask=mask_nodes, grid=grid))
        snr.append(stats["snr"])

    run = invert(theta0, LIMB, N, sources, observations,
                 num_cycles=NUM_CYCLES, max_iters=MAX_ITERS, ftol=FTOL, gtol=GTOL)

    return {"percent": percent, "seed": seed,
            "sigma": run.sigma, "interfaces": run.interfaces,
            "final_loss": float(run.loss_history[-1]),
            "n_evaluations": run.n_evaluations, "n_iterations": run.n_iterations,
            "n_restarts": run.n_restarts, "message": run.message,
            "grad_max": run.grad_max, "snr": tuple(snr), "pid": os.getpid()}

In [11]:
# the sweep. joblib's loky backend starts fresh processes and serialises the
# function with cloudpickle, so a function defined in this notebook is dispatchable
# without being moved into a module - and fresh processes rather than forks, which
# is what makes this safe to run from a kernel that has already loaded a threaded
# BLAS. Results come back in the order the cases were submitted; they are keyed on
# the case anyway, so nothing below depends on that.

cases = [(percent, seed)
         for percent in NOISE_PERCENTS
         for seed in draws_for(percent)]

t0 = time.time()
completed = Parallel(n_jobs=N_WORKERS, backend="loky", verbose=10)(
    delayed(invert_one)(percent, seed, theta_start, phi_clean)
    for percent, seed in cases
)
elapsed = time.time() - t0

results = {(r["percent"], r["seed"]): r for r in completed}
evaluations = sum(r["n_evaluations"] for r in completed)

print(f"\n{len(results)} cases in {elapsed / 60:.1f} min on {N_WORKERS} workers")
print(f"{evaluations} objective evaluations, "
      f"{elapsed * N_WORKERS / evaluations:.2f} s per evaluation per worker")
print(f"{len(set(r['pid'] for r in completed))} distinct worker processes")

[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done   2 tasks      | elapsed: 60.4min
[Parallel(n_jobs=3)]: Done   7 tasks      | elapsed: 153.0min
[Parallel(n_jobs=3)]: Done  13 out of  16 | elapsed: 278.5min remaining: 64.3min



16 cases in 308.9 min on 3 workers
4945 objective evaluations, 11.24 s per evaluation per worker
3 distinct worker processes


[Parallel(n_jobs=3)]: Done  16 out of  16 | elapsed: 308.9min finished


In [12]:
# how each case ended. Two things are asked, and neither is the iteration count:
# `message` says whether the last call stopped on its own criteria or ran out of the
# budget, and max|g| says whether the point is first-order stationary at all. A count
# cannot answer either once invert restarts, because MAX_ITERS is a per-call option
# and n_iterations is a sum across calls.
#
# max|g| is printed rather than only tested. The gradient here is inexact - the
# frozen diagonal in probe_diagonal, and a forward solve on a fixed cycle budget - so
# it has a floor under it that restarting cannot get below, and where that floor sits
# is a result worth reading rather than a pass or a fail. The control run is the
# clearest look at it: with no noise in the data, what is left is all solver.

print(f"{'noise':>7} {'draw':>5} {'SNR':>8} {'iters':>7} {'restarts':>9} "
      f"{'max|g|':>10} {'final J':>11}   exit")
for percent, seed in cases:
    r = results[percent, seed]
    snr = np.mean(r["snr"])
    print(f"{percent:>6.2f}% {seed:>5} {'inf' if not np.isfinite(snr) else f'{snr:.1f}':>8} "
          f"{r['n_iterations']:>7} {r['n_restarts']:>9} {r['grad_max']:>10.2e} "
          f"{r['final_loss']:>11.4e}   {r['message']}")

stalled = [(p, s) for p, s in cases
           if results[p, s]["message"].startswith("STOP") or results[p, s]["grad_max"] > GTOL]
if stalled:
    print(f"\nWARNING: {len(stalled)} of {len(cases)} cases did not reach "
          f"max|g| <= {GTOL:g} - those iterates are not converged minimisers")
    print("         " + ", ".join(f"{p:g}%/draw {s}" for p, s in stalled))

  noise  draw      SNR   iters  restarts     max|g|     final J   exit
  0.00%     0      inf     200        41   1.72e-12  7.2118e-18   STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
  2.00%     0     50.0     127         6   3.62e-10  1.6773e-06   ABNORMAL: 
  2.00%     1     50.1     178        80   9.92e-11  1.6690e-06   CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  2.00%     2     49.9     174         7   4.15e-10  1.6996e-06   CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  2.00%     3     50.1     174        37   3.83e-10  1.6952e-06   CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  2.00%     4     49.9     131         3   8.81e-11  1.6937e-06   CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  3.00%     0     33.4     138         4   4.81e-10  3.7739e-06   CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  3.00%     1     33.4     101         3   1.12e-10  3.7554e-06   CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  3.00%     2     33.3     1

In [13]:
# the reduction. Every case contributes eight numbers - five conductivities and the
# three recovered layer thicknesses - so the sweep is a (noise level, draw, quantity)
# array, ragged in its middle axis because the control has one draw and the noisy
# levels have five, and every output below is a different pair of axes collapsed out
# of it. It is held as a list of per-level arrays rather than padded into a
# rectangle: padding the control out to five would put four copies of one answer into
# every average and spread taken over that level.
#
# The relative error is formed per run and then averaged, not reconstructed from the
# averaged values afterwards: the mean of a ratio is not the ratio of the means, and
# for a parameter approached from both sides across the draws the two differ by a
# lot. It is the mean of the errors that says how wrong a typical realisation was.

sigma_ref = true_sigma(LIMB)
thick_ref = thicknesses(LIMB.interfaces, scale=MM_PER_UNIT)[:-1]
reference = torch.cat([sigma_ref, thick_ref])

QUANTITIES = ([f"{nm} (S/m)" for nm in param_names]
              + [f"{nm} (mm)" for nm in layer_names])
# a conductivity is quoted to four decimals and a thickness to two: a tenth of a
# micron of muscle is not a claim this experiment is making
DECIMALS = [".4f"] * len(param_names) + [".2f"] * len(layer_names)
IS_SIGMA = torch.tensor([True] * len(param_names) + [False] * len(layer_names))

recovered = [
    torch.stack([torch.cat([results[percent, seed]["sigma"],
                            thicknesses(results[percent, seed]["interfaces"],
                                        scale=MM_PER_UNIT)[:-1]])
                 for seed in draws_for(percent)])
    for percent in NOISE_PERCENTS
]                                                   # one (draws, quantity) per level
rel_error = [(level - reference).abs() / reference for level in recovered]



def spread(values):
    """Sample standard deviation down the draw axis, nan where there is one draw.

    The n - 1 correction, because the draws are a sample of the noise realisations
    the generator could have produced and not the population of interest. torch's
    own answer at a single sample is nan, which is the honest one - one run says
    nothing about spread - but it arrives with a warning about the degrees of
    freedom, and the control level is a single run by design rather than by
    accident. Asked here, the nan is the intent.
    """
    if values.shape[0] > 1:
        return values.std(0, correction=1)
    return torch.full_like(values[0], float("nan"))


# stacked over the levels once the draw axis is gone, which is the shape every table
# and the figure want
mean_recovered = torch.stack([level.mean(0) for level in recovered])
std_recovered = torch.stack([spread(level) for level in recovered])
mean_error = torch.stack([level.mean(0) for level in rel_error])

# the per-draw aggregates, kept ragged because their spread is taken over draws
per_draw_all = [level.mean(1) for level in rel_error]
per_draw_sigma = [level[:, IS_SIGMA].mean(1) for level in rel_error]
per_draw_thick = [level[:, ~IS_SIGMA].mean(1) for level in rel_error]

CONTROL = NOISE_PERCENTS.index(0.0)
NOISY = [i for i, percent in enumerate(NOISE_PERCENTS) if percent > 0.0]

print(f"{len(NOISE_PERCENTS)} levels x "
      + "/".join(str(len(draws_for(p))) for p in NOISE_PERCENTS)
      + f" draws x {len(QUANTITIES)} quantities = {N_CASES} runs")
print(f"noise-free floor, averaged over the {len(QUANTITIES)} quantities: "
      f"{float(mean_error[CONTROL].mean()):.2%}")

4 levels x 1/5/5/5 draws x 8 quantities = 16 runs
noise-free floor, averaged over the 8 quantities: 0.00%


In [14]:
# save the sweep's numbers to experiment-results, flat and lossless.
#
# 16 rows keyed by (noise, draw) rather than a (level, draw, quantity) block: the
# draw axis is ragged - one control run at 0% and five draws at each of the three
# noisy levels - and padding the control out to five would put four copies of one
# answer into anything later averaged over that level. The keys are there to
# reshape on the way back in if a rectangle is what you want.

import numpy as np

rows = list(cases)

sigma = np.stack([results[k]["sigma"].numpy() for k in rows])            # (16, 5) S/m
radii = np.stack([results[k]["interfaces"].numpy() for k in rows])       # (16, 3) normalised
thick = np.stack([thicknesses(results[k]["interfaces"], scale=MM_PER_UNIT)[:-1].numpy()
                  for k in rows])                                       # (16, 3) mm
quantity = np.concatenate([sigma, thick], axis=1)                       # (16, 8) as the tables read
relative_error = np.abs(quantity - reference.numpy()) / reference.numpy()

path = RESULTS / f"measurement-noise-sweep{OUTPUT_SUFFIX}.npz"
np.savez_compressed(
    path,
    noise_percent=np.array([p for p, _ in rows]),
    draw=np.array([s for _, s in rows]),
    quantity_names=np.array(QUANTITIES),
    quantity=quantity,
    relative_error=relative_error,
    reference=reference.numpy(),
    sigma=sigma,
    interfaces=radii,
    thickness_mm=thick,
    final_loss=np.array([results[k]["final_loss"] for k in rows]),
    grad_max=np.array([results[k]["grad_max"] for k in rows]),
    n_iterations=np.array([results[k]["n_iterations"] for k in rows]),
    n_evaluations=np.array([results[k]["n_evaluations"] for k in rows]),
    message=np.array([results[k]["message"] for k in rows]),
    snr=np.array([results[k]["snr"] for k in rows]),
)
print(f"{quantity.shape[0]} runs x {quantity.shape[1]} quantities -> {path.resolve()}")
print("levels " + "/".join(f"{p:g}%" for p in NOISE_PERCENTS)
      + ", draws " + "/".join(str(len(draws_for(p))) for p in NOISE_PERCENTS))


16 runs x 8 quantities -> ./experiment-results/measurement-noise-sweep-2-4pc.npz
levels 0%/2%/3%/4%, draws 1/5/5/5
